# 2 · Mensajes y cadena

Lección 4 del módulo 1: [Cadena](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58238466-lesson-4-chain). El notebook original es [`chain.ipynb`](../../langchain-academy/module-1/chain.ipynb).

Aquí entra un modelo de lenguaje. Se juntan cuatro ideas:

- los **mensajes**, que son la forma de hablar con un modelo de chat;
- las **herramientas**, funciones que el modelo puede pedir que se ejecuten;
- los mensajes como **estado** del grafo;
- el **reducer**, que hace que los mensajes se acumulen en vez de reemplazarse.

## El modelo

Estos apuntes usan un modelo local con [Ollama](https://ollama.com), gratis y sin claves: `qwen2.5:7b`, que sabe llamar herramientas. Antes de ejecutar, con Ollama abierto:

```
ollama pull qwen2.5:7b
```

`temperature=0` hace que el modelo elija siempre la opción más probable, así las respuestas casi no cambian de una ejecución a otra.

El curso usa `gpt-4o` de OpenAI. Para usarlo, cambia la celda de abajo por:

```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o", temperature=0)
```

El resto del notebook queda igual: al grafo no le importa qué modelo hay detrás.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:7b", temperature=0)

## Mensajes

Un modelo de chat no recibe un texto suelto sino una lista de mensajes, cada uno con un rol:

| Tipo | Quién lo escribe |
|---|---|
| `SystemMessage` | Las instrucciones generales para el modelo |
| `HumanMessage` | La persona |
| `AIMessage` | El modelo |
| `ToolMessage` | Una herramienta, con el resultado de una llamada |

Cada mensaje tiene un `content` y, si se quiere, un `name` que identifica quién habla.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

mensajes = [AIMessage(content="¿Así que estás investigando sobre mamíferos marinos?", name="Modelo")]
mensajes.append(HumanMessage(content="Sí, así es.", name="Benja"))
mensajes.append(AIMessage(content="Genial, ¿qué te gustaría saber?", name="Modelo"))
mensajes.append(HumanMessage(content="¿Dónde se pueden ver ballenas en Chile?", name="Benja"))

for m in mensajes:
    m.pretty_print()

## Llamar al modelo

`invoke` recibe la lista completa y devuelve un `AIMessage` con la respuesta. El modelo no recuerda nada entre una llamada y otra: la conversación entera viaja en cada `invoke`.

In [ ]:
resultado = llm.invoke(mensajes)
type(resultado)

In [ ]:
print(resultado.content)

Además del texto, la respuesta dice cuántos tokens leyó el modelo y cuántos escribió:

In [ ]:
resultado.usage_metadata

## Herramientas

Una herramienta es una función que el modelo puede pedir que se ejecute. El modelo nunca la ejecuta: solo responde algo como «quiero llamar a `multiplicar` con `a=2` y `b=3`». Ejecutarla le toca al grafo, y eso es el notebook 3.

Para que el modelo sepa qué herramientas hay, LangChain convierte cada función en una descripción con su nombre, su docstring y los tipos de sus argumentos. Por eso el docstring importa: es lo que el modelo lee para decidir si la usa.

In [ ]:
def multiplicar(a: int, b: int) -> int:
    """Multiplica a por b.

    Args:
        a: primer entero
        b: segundo entero
    """
    return a * b

Esta es la descripción que recibe el modelo:

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

convert_to_openai_tool(multiplicar)

`bind_tools` crea una versión del modelo que manda esa descripción en cada llamada:

In [ ]:
llm_con_herramientas = llm.bind_tools([multiplicar])

llamada_herramienta = llm_con_herramientas.invoke([HumanMessage(content="¿Cuánto es 2 multiplicado por 3?", name="Benja")])
llamada_herramienta.tool_calls

La respuesta no trae texto sino una **llamada a herramienta**: el nombre de la función, sus argumentos y un `id`. Nadie ejecutó `multiplicar` todavía.

## Mensajes como estado

Para usar mensajes en un grafo, el estado necesita una clave con la lista de mensajes:

In [ ]:
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict


class SinReducer(TypedDict):
    messages: list[AnyMessage]

Pero hay un problema. Como se vio en el notebook 1, lo que devuelve un nodo **reemplaza** al valor anterior. Si un nodo devuelve `{"messages": [respuesta]}`, la conversación anterior se pierde. Este grafo, sin modelo, lo muestra con dos nodos que agregan un mensaje cada uno:

In [ ]:
from langgraph.graph import END, START, StateGraph


def nodo_a(estado):
    return {"messages": [AIMessage(content="mensaje de a")]}

def nodo_b(estado):
    return {"messages": [AIMessage(content="mensaje de b")]}


def dos_nodos(esquema):
    constructor = StateGraph(esquema)
    constructor.add_node("nodo_a", nodo_a)
    constructor.add_node("nodo_b", nodo_b)
    constructor.add_edge(START, "nodo_a")
    constructor.add_edge("nodo_a", "nodo_b")
    constructor.add_edge("nodo_b", END)
    return constructor.compile()


final = dos_nodos(SinReducer).invoke({"messages": [HumanMessage(content="entrada")]})
[m.content for m in final["messages"]]

Solo sobrevive el último mensaje.

## Reducers

Un **reducer** es una función que dice cómo combinar el valor nuevo con el anterior. Se agrega a la clave con `Annotated`. `add_messages` es el reducer para mensajes: agrega los nuevos al final de la lista.

In [ ]:
from typing import Annotated

from langgraph.graph.message import add_messages


class ConReducer(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


final = dos_nodos(ConReducer).invoke({"messages": [HumanMessage(content="entrada")]})
[m.content for m in final["messages"]]

Ahora se conservan los tres. `add_messages` también se puede probar solo, fuera de un grafo:

In [ ]:
anteriores = [
    AIMessage(content="¡Hola! ¿En qué te ayudo?", name="Modelo"),
    HumanMessage(content="Busco información sobre biología marina.", name="Benja"),
]
nuevo = AIMessage(content="Claro, ¿qué te interesa en particular?", name="Modelo")

add_messages(anteriores, nuevo)

Cada mensaje recibió un `id`. `add_messages` los usa: si llega un mensaje con el mismo `id` que uno que ya está, lo **reemplaza** en vez de agregarlo. En el módulo 2 eso sirve para editar y borrar mensajes.

In [ ]:
original = AIMessage(content="primera versión", id="respuesta-1")
corregido = AIMessage(content="versión corregida", id="respuesta-1")

[m.content for m in add_messages([original], [corregido])]

Un estado con una lista de mensajes es tan común que LangGraph trae uno hecho: `MessagesState`, con la clave `messages` y el reducer `add_messages` ya puestos. Si hacen falta más claves, se crea una clase que herede de él y se agregan ahí.

## El grafo

Con todo eso, el grafo de esta lección tiene un solo nodo: llama al modelo con herramientas usando los mensajes del estado y devuelve la respuesta, que el reducer agrega a la conversación.

In [ ]:
from IPython.display import Image, display
from langgraph.graph import MessagesState


def llamar_llm(estado: MessagesState):
    return {"messages": [llm_con_herramientas.invoke(estado["messages"])]}


constructor = StateGraph(MessagesState)
constructor.add_node("llamar_llm", llamar_llm)
constructor.add_edge(START, "llamar_llm")
constructor.add_edge("llamar_llm", END)
grafo = constructor.compile()

display(Image(grafo.get_graph().draw_mermaid_png()))

Con un saludo, el modelo responde con texto:

In [ ]:
resultado = grafo.invoke({"messages": HumanMessage(content="¡Hola!")})
for m in resultado["messages"]:
    m.pretty_print()

Se pasó un solo `HumanMessage` y no una lista: `add_messages` acepta las dos formas. Si se le pide una multiplicación, el modelo decide usar la herramienta:

In [ ]:
resultado = grafo.invoke({"messages": HumanMessage(content="Multiplica 2 por 3")})
for m in resultado["messages"]:
    m.pretty_print()

El grafo termina con la llamada a herramienta, pero nadie la ejecutó: falta un nodo que lo haga. Eso es el **router** del notebook 3.

## Resumen

- Los modelos de chat reciben y devuelven **mensajes** con roles.
- Una **herramienta** es una función descrita por su nombre, su docstring y sus tipos. El modelo pide usarla con una llamada a herramienta (`tool_calls`), pero no la ejecuta.
- Lo que devuelve un nodo reemplaza al valor anterior, salvo que la clave tenga un **reducer**. `add_messages` agrega los mensajes nuevos y reemplaza los que tienen el mismo `id`.
- `MessagesState` es un estado con `messages` y `add_messages` ya listos.

## Ejercicios

1. Agrega una herramienta `sumar(a, b)` y pregunta «¿Cuánto es 7 más 5?». ¿Qué herramienta eligió el modelo?
2. Borra el docstring de `multiplicar` y vuelve a ejecutar `convert_to_openai_tool(multiplicar)`. ¿Qué cambió en la descripción? ¿El modelo la sigue eligiendo bien?
3. Arma un estado con una clave `total: Annotated[int, operator.add]` (hay que importar `operator`) y haz que `nodo_a` y `nodo_b` devuelvan `{"total": 1}`. ¿Cuánto vale `total` al final si la entrada es `{"total": 0}`?